# Lab 13 — Dashboards, Streaming, and Deployment

## Part 1 — Install Required Libraries

In [1]:
# Install Dash stack (skip if already installed)
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "dash>=2.17.0",
    "dash-bootstrap-components>=1.6.0",
    "pymongo>=4.6.0",
    "gunicorn>=22.0.0",
    "websockets>=12.0.0",
], check=True)
print("All packages ready.")

All packages ready.


In [2]:
# Verify imports
import dash
import dash_bootstrap_components as dbc
import pymongo
import plotly
import pandas as pd

print(f"dash                     {dash.__version__}")
print(f"dash-bootstrap-components {dbc.__version__}")
print(f"pymongo                  {pymongo.__version__}")
print(f"plotly                   {plotly.__version__}")
print(f"pandas                   {pd.__version__}")

/Users/amilacausevic/Desktop/Movie-Industry-Analytics-Pipeline/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


dash                     4.1.0
dash-bootstrap-components 2.0.4
pymongo                  4.16.0
plotly                   6.7.0
pandas                   3.0.2


## Part 2 — Data Access Layer

All database queries are isolated in `src/dashboard/data_access.py`. This keeps callbacks clean and makes the data layer independently testable.

The module reads a `MONGO_URI` environment variable, so the same code works locally (`mongodb://localhost:27017`) and inside Docker Compose (`mongodb://db:27017`).

In [1]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))

from dashboard.data_access import load_articles_df, get_categories, get_years, filter_articles

df = load_articles_df()
print(f"Shape  : {df.shape}")
print(f"Columns: {list(df.columns)}")
df[['source_name', 'title', 'Category', 'Word Count', 'published_year']].head(3)

Shape  : (297, 23)
Columns: ['source_name', 'title', 'author', 'description', 'publishedAt', 'content', 'text', 'ID', 'Title', 'Source', 'Author', 'Published Date', 'Word Count', 'Category', 'Nutrition', 'Pharmacology', 'Mental Health', 'Total', 'raw_text', 'processed_text', 'published_date', 'image_url', 'published_year']


,source_name,title,Category,Word Count,published_year
0,Science Daily,Scientists discover diet that tricks the body ...,NaN,NaN,2026
1,Naturalnews.com,Nature’s powerhouses: The top 8 healthiest ber...,NaN,NaN,2026
2,Eatingwell.com,6 Things You Should Do After 5 P.M. to Support...,NaN,NaN,2026


In [2]:
categories = get_categories(df)
year_min, year_max = get_years(df)

print(f"Categories ({len(categories)}): {categories}")
print(f"Year range: {year_min} – {year_max}")

Categories (0): []
Year range: 2026 – 2026


### Demonstrating `filter_articles()`

All three filter parameters are optional. Passing `None` skips that filter entirely.

In [3]:
first_cat = categories[0] if categories else None

# Filter by category only
cat_df = filter_articles(df, category=first_cat)
print(f"'{first_cat}' articles: {len(cat_df)}")

# Filter by category + year range
recent = filter_articles(df, category=first_cat, year_range=(2025, 2026))
print(f"'{first_cat}' 2025-2026: {len(recent)}")

# Filter by title search
searched = filter_articles(df, search="health")
print(f"Titles containing 'health': {len(searched)}")
searched[["title", "published_year", "Category"]].head()

'None' articles: 297
'None' 2025-2026: 297
Titles containing 'health': 35


,title,published_year,Category
1,Nature’s powerhouses: The top 8 healthiest ber...,2026,NaN
2,6 Things You Should Do After 5 P.M. to Support...,2026,NaN
9,8 Hidden Health Benefits of Green Tea,2026,NaN
11,A tangy brew for metabolic health: Kombucha sh...,2026,NaN
15,Is It Healthy To Eat A Banana Every Day? Here'...,2026,NaN


## Part 3 — Understanding the Dash Framework

Every Dash application has four ordered parts:

1. **Create the app object** — `Dash(__name__, external_stylesheets=[...])`
2. **Define the layout** — a tree of Python objects that Dash translates to HTML
3. **Register callbacks** — `@app.callback(Output(...), Input(...))` decorators
4. **Run the server** — `app.run(debug=True)` for development; Gunicorn in production

### Layout building blocks

| Module | Role |
|--------|------|
| `dash.html` | One Python class per HTML tag (`html.Div`, `html.H1`, …) |
| `dash.dcc`  | Higher-level interactive components (`dcc.Dropdown`, `dcc.Graph`, `dcc.Interval`) |
| `dash_bootstrap_components` | Responsive grid, cards, navbars |

### Callback anatomy

```python
@app.callback(
    Output("revenue-chart", "figure"),    # what gets updated
    Input("genre-filter",   "value"),     # what triggers the update
)
def update_chart(selected_category):
    df = filter_articles(_DF, category=selected_category)
    return px.bar(df, x="source_name", y="count")
```

When the user changes the dropdown, Dash automatically re-runs `update_chart` and pushes the new figure to the browser — no page reload needed.

## Part 4 — Chart Demonstrations

Each chart type is justified by the nature of the data:

| Chart | Type | Justification |
|-------|------|---------------|
| Top 10 Sources | Horizontal bar | Long source names read left-to-right (Tufte: match chart to data) |
| Articles by Category | Bar chart | Compares discrete category counts directly |
| Word Count by Category | Box plot | Shows full distribution: median, IQR, outliers — not just the mean |
| Articles per Year | Line | Trends over a continuous time axis are best shown as a line |

In [4]:
import plotly.express as px
import plotly.graph_objects as go

DARK_TEMPLATE = "plotly_dark"
CHART_BG = "#112236"

# ── Chart 1: Top 10 Sources by Article Count ──────────────────────────────
top10_sources = (
    df.groupby("source_name")
    .size()
    .nlargest(10)
    .reset_index(name="count")
)

fig1 = px.bar(
    top10_sources, x="count", y="source_name", orientation="h",
    text="count",
    color="count", color_continuous_scale="Blues",
    template=DARK_TEMPLATE,
    title="Top 10 Sources by Article Count",
    labels={"count": "Article Count", "source_name": ""},
)
fig1.update_traces(textposition="outside")
fig1.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG,
    coloraxis_showscale=False,
    yaxis={"categoryorder": "total ascending"},
)
fig1.show()

In [5]:
# ── Chart 2: Article Count by Category (Bar Chart) ──────────────────────
cat_counts = df["Category"].dropna().value_counts().reset_index()
cat_counts.columns = ["Category", "count"]

fig2 = px.bar(
    cat_counts, x="Category", y="count", color="Category",
    template=DARK_TEMPLATE,
    title="Article Count by Category",
    labels={"count": "Article Count", "Category": ""},
)
fig2.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG, showlegend=False,
)
fig2.show()

In [ ]:
# ── Chart 3: Word Count Distribution by Category (Box Plot) ─────────────
import pandas as pd

wc_df = df.copy()
wc_df["Word Count"] = pd.to_numeric(wc_df["Word Count"], errors="coerce")
wc_df = wc_df[wc_df["Word Count"] > 0].dropna(subset=["Word Count", "Category"])

fig3 = px.box(
    wc_df, x="Category", y="Word Count", color="Category",
    template=DARK_TEMPLATE,
    title="Word Count Distribution by Category",
    labels={"Word Count": "Word Count", "Category": ""},
)
fig3.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG, showlegend=False,
)
fig3.show()

In [ ]:
# ── Chart 4: Articles Published per Year (Line Chart) ────────────────────
yearly = df.groupby("published_year").size().reset_index(name="count")
yearly = yearly[yearly["published_year"] > 2000]

fig4 = px.line(
    yearly, x="published_year", y="count", markers=True,
    template=DARK_TEMPLATE,
    title="Articles Published per Year",
    labels={"published_year": "Year", "count": "Articles Published"},
)
fig4.update_traces(line_color="#4a9eff", marker_color="#34d399")
fig4.update_layout(paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG)
fig4.show()

In [ ]:
# ── Chart 5: Live Ingestion Ticker (simulated — static snapshot) ─────────
import random, time, collections

buffer = collections.deque(maxlen=60)
for i in range(20):
    buffer.append({"tick": i, "value": 50 + random.gauss(0, 5) + 0.3 * (i % 20)})

fig5 = go.Figure(go.Scatter(
    x=[p["tick"] for p in buffer],
    y=[p["value"] for p in buffer],
    mode="lines+markers",
    line=dict(color="#34d399", width=2),
    fill="tozeroy", fillcolor="rgba(52,211,153,0.1)",
))
fig5.update_layout(
    template=DARK_TEMPLATE, paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG,
    title="Live Ingestion Rate (simulated — 20 ticks)",
    xaxis_title="Tick", yaxis_title="Articles / sec",
    height=250,
)
fig5.show()

## Part 5 — Callback Patterns

| Pattern | Callback | Trigger | Output |
|---------|----------|---------|--------|
| N → 1 | Top sources bar | category + year + search | `revenue-chart` figure |
| N → 1 | Category bar | category + year + search | `rating-chart` figure |
| N → 1 | Word count box | category + year + search | `scatter-chart` figure |
| N → 1 | Trend line | category + year + search | `trend-chart` figure |
| 1 → 1 | Live ticker | `dcc.Interval` tick | `live-chart` figure |

### Live Ticker Design

`dcc.Interval` fires a callback every 3000 ms. The callback appends a randomly-generated data point to a `collections.deque(maxlen=60)` module-level buffer. Because the deque has a fixed maximum length, old points are automatically discarded — no unbounded memory growth.

In a real production system the callback would read from:
- A WebSocket stream (e.g. live news ingestion API)
- A time-series database such as InfluxDB
- A message queue such as Kafka

## Part 6 — Dockerfile Walkthrough

In [11]:
dockerfile_path = os.path.join("..", "Dockerfile")
with open(dockerfile_path) as f:
    print(f.read())

# Dockerfile
# ─────────────────────────────────────────────────────────────
# Single-stage build — python:3.12-slim (Debian, minimal footprint).
# ─────────────────────────────────────────────────────────────

FROM python:3.12-slim

LABEL maintainer="student@ibu.edu.ba"
LABEL description="Movie Industry Analytics — Dash Dashboard"

WORKDIR /app

# ── Layer 1: dependencies ─────────────────────────────────────
# Copy requirements first so Docker can cache this layer.
# pip install is skipped on subsequent builds unless
# requirements.txt changes.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Layer 2: application code ─────────────────────────────────
COPY . .

# Port the Dash server listens on (documentation only — the
# actual binding is done by Gunicorn via CMD).
EXPOSE 8050

# ── Runtime ───────────────────────────────────────────────────
# Gunicorn replaces the Flask dev server in production:
#   --workers 2     two processes (safe for low-memory 

### Dockerfile instruction explanations

| Instruction | Purpose |
|-------------|----------|
| `FROM python:3.12-slim` | Minimal Debian image with Python 3.12 — smaller than the full image |
| `WORKDIR /app` | All subsequent commands run in `/app` inside the container |
| `COPY requirements.txt .` | Copy only the requirements file first to enable layer caching |
| `RUN pip install --no-cache-dir` | Install deps; `--no-cache-dir` keeps the image smaller |
| `COPY . .` | Copy application code (separate layer from deps) |
| `EXPOSE 8050` | Documents the port (does not actually publish it) |
| `CMD ["gunicorn", ...]` | Production WSGI server — multi-worker, no hot-reload |

**Why Gunicorn instead of `app.run()`?**  
`app.run(debug=True)` runs a single-threaded Flask development server with hot-reload enabled. Gunicorn forks multiple worker processes and has no development overhead — it is the correct production choice.

## Part 7 — Running the Dashboard Locally

In [7]:
# Verify app.py can be imported without errors
import importlib.util, pathlib

app_path = pathlib.Path("..") / "app.py"
spec = importlib.util.spec_from_file_location("app", app_path)
app_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_module)

print("app.py imported successfully.")
print(f"Dash app title : {app_module.app.title}")
print(f"Flask server   : {type(app_module.server).__name__}")

app.py imported successfully.
Dash app title : Health & Wellness Dashboard
Flask server   : Flask


To start the dashboard in development mode, run from the project root:

```bash
python app.py
```

Then open **http://127.0.0.1:8050** in a browser.

## Part 8 — MongoDB Seeding

In [8]:
# Show the seed script contents
seed_path = os.path.join("..", "scripts", "seed_mongo.py")
with open(seed_path) as f:
    print(f.read())

"""
One-time script: load the cleaned articles CSV into MongoDB.

Run once before starting the dashboard:
    python scripts/seed_mongo.py
"""
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))

import pandas as pd
from pymongo import MongoClient

MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.environ.get("MONGO_DB", "articles_pipeline")

_BASE = os.path.join(os.path.dirname(__file__), "..", "data", "processed", "cleaned")
CSV_PATH = None
for _name in ("articles_clean.csv", "articles_cleaned.csv"):
    _candidate = os.path.join(_BASE, _name)
    if os.path.exists(_candidate):
        CSV_PATH = _candidate
        break

if CSV_PATH is None:
    CSV_PATH = os.path.join(_BASE, "articles_clean.csv")  # will error with clear message


def seed():
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df):,} rows from {CSV_PATH}")

    client = MongoClient(MONGO_URI)
    col = client[DB_NAME]["raw_articles"]
    

In [9]:
# Run the seed script (will fail gracefully if MongoDB is not running)
import subprocess
result = subprocess.run(
    [sys.executable, seed_path],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("[Note] MongoDB not running — dashboard will fall back to CSV.")
    print(result.stderr[:300])

Loaded 297 rows from /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/notebooks/../scripts/../data/processed/cleaned/articles_clean.csv
Inserted 297 documents into articles_pipeline.raw_articles
Indexes created.



## Part 9 — Deployment Documentation

### Prerequisites

- **Docker Desktop** (v4.x or later) installed and running on your machine.
- Ports **8050** and **27017** not in use by other applications.
- The project repository cloned locally.
- Python 3.11+ and `pip` available (only needed for the seed script and local dev).

---

### Step-by-Step Deployment

**Step 1: Build and start the full stack**

```bash
docker compose up --build
```

This command:
1. Builds the Dash app image from the local `Dockerfile`.
2. Pulls the `mongo:7` image from Docker Hub.
3. Starts both services on a private Docker network.

Wait until you see `Booting worker with pid` in the logs — the app is then ready.

To run in the background (detached mode):

```bash
docker compose up --build -d
docker compose logs -f          # stream logs
```

---

**Step 2: Seed MongoDB (first run only)**

```bash
python scripts/seed_mongo.py
```

This loads `data/processed/cleaned/articles_clean.csv` into the `articles_pipeline.raw_articles` MongoDB collection and creates indexes on `source_name`, `published_year`, `title`, and `Category`.

> Run this only once. Subsequent starts do **not** need re-seeding unless you run `docker compose down -v` (which deletes the named volume).

---

**Step 3: Open the dashboard**

Navigate to **http://localhost:8050** in any browser.

---

**Step 4: Verify functionality**

- [ ] KPI cards display total articles, average word count, unique sources, and category count.
- [ ] Change the **Category** dropdown — all four charts update instantly.
- [ ] Drag the **Year Range** slider — charts filter to that time window.
- [ ] Type a keyword in the **Search** box — charts filter to matching articles.
- [ ] The **Live Ticker** updates automatically every 3 seconds.

---

**Step 5: Stop the stack**

```bash
docker compose down        # stop containers, keep database volume
docker compose down -v     # also delete the mongo-data volume
```

---

### Troubleshooting

| Problem | Solution |
|---------|----------|
| Port 8050 already in use | Change the host port in `docker-compose.yml`: `"8051:8050"` |
| Port 27017 already in use | Change: `"27018:27017"` and update `MONGO_URI` accordingly |
| MongoDB connection refused in seed script | The stack may still be starting — wait for the healthcheck to pass (`docker compose ps`), then re-run the seed script |
| Charts show "No data" / empty | Run `python scripts/seed_mongo.py`; verify CSV exists at `data/processed/cleaned/articles_clean.csv` |
| Image build fails | Clear cached layers: `docker compose build --no-cache` |
| `gunicorn: command not found` | Gunicorn must be in `requirements.txt` — confirm with `grep gunicorn requirements.txt` |

## Part 10 — Assignment 13 Solution Summary

### What was built

| Component | File | Deliverable |
|-----------|------|-------------|
| Dash Application (35%) | `app.py`, `src/dashboard/layout.py` | Full dark-themed layout with 3 interactive controls and 4+1 charts |
| Callbacks (25%) | `src/dashboard/callbacks.py` | 5 callbacks (4 filter + 1 live ticker with `dcc.Interval`) |
| MongoDB Integration (15%) | `src/dashboard/data_access.py`, `scripts/seed_mongo.py` | Reads from MongoDB `articles_pipeline.raw_articles`, falls back to CSV gracefully |
| Docker Deployment (15%) | `Dockerfile`, `docker-compose.yml`, `.dockerignore` | Single `docker compose up --build` starts full stack |
| Deployment Documentation (10%) | This notebook (Part 9 above) | Prerequisites, step-by-step commands, verification checklist, troubleshooting table |

### Visualisation design decisions (Tufte principles)

1. **Horizontal bar chart** for top sources — long source names fit naturally on the horizontal axis, avoiding diagonal label rotation that hides data.
2. **Bar chart** for category counts — discrete categories with no natural ordering are best compared with aligned bars; length encodes quantity directly.
3. **Box plot** for word count distribution — shows median, interquartile range, and outliers simultaneously. A simple bar of means would hide the variance in article length across categories.
4. **Line chart** for yearly trend — a line communicates continuity between adjacent years, which is appropriate for a time series. Bars would imply the values are independent categories.
5. **Dark theme throughout** — consistent `plotly_dark` template with `#112236` panel backgrounds creates a cohesive dashboard feel and reduces eye strain for the audience of stakeholders using the tool in low-light environments.

### Key architectural decisions

- **`server = app.server`** in `app.py` exposes the underlying Flask application so Gunicorn can serve it in production without any code change.
- **Environment variable for `MONGO_URI`** — the same application binary works locally (defaults to `localhost:27017`) and inside Docker Compose (receives `mongodb://db:27017` from `docker-compose.yml`).
- **`collections.deque(maxlen=60)`** for the live ticker — a fixed-capacity circular buffer guarantees constant memory regardless of how long the dashboard runs.
- **CSV fallback in `load_articles_df()`** — if MongoDB is unavailable the application degrades gracefully and still shows all four filter-driven charts.